# День 3 (v2) — дообучение бейзлайна на всех данных
Что изменилось по сравнению с первой версией:
- `baseline.onnx` и данные скачиваются **прямо в Colab** из официального стартового пакета (~6 мин). Ничего не нужно загружать на Drive вручную — из-за этого и упала первая версия.
- Обучение на **всех 10 607** последовательностях train, а не на 3500: `train.py` читает их прямо из `train.parquet`, без конвертации.
- Сохраняется каждая эпоха, в сабмит идёт последняя (почему — в ячейке 5).

Перед запуском: **Среда выполнения → Сменить среду выполнения → T4 GPU**. Обновлённая папка `day2` должна лежать на Drive в `Мой диск/wunder/day2`.

## 1. GPU + Drive + код

In [ ]:
!nvidia-smi --query-gpu=name --format=csv,noheader
from google.colab import drive
drive.mount('/content/drive')
W = '/content/drive/MyDrive/wunder'
!pip -q install onnx
!rm -rf /content/day2 && cp -r $W/day2 /content/day2
%cd /content/day2
!python -m pytest -q tests/test_day2.py -k 'loss or chunked or parquet_source or reader_error'

## 2. Стартовый пакет прямо в Colab (~6 мин)
Скачиваем официальный архив (31 ГБ, ~200 МБ/с), распаковываем только `baseline.onnx`, `train.parquet` и `valid.parquet`, архив удаляем. Копия `baseline.onnx` кладётся на Drive.

Если скачивание не работает (ссылка поменялась): загрузите `baseline.onnx` из стартового пакета в `wunder/` на Drive и запустите старый `day3_finetune.ipynb`.

In [ ]:
import glob, hashlib, os, shutil
URL = 'https://files.wundernn.io/wnn_connectome_starterpack.tar.gz'
!apt-get -qq update > /dev/null 2>&1; apt-get -qq install -y aria2 > /dev/null 2>&1
!aria2c -c -x 16 -s 16 -k 1M --file-allocation=none --summary-interval=30 --console-log-level=warn -d /content -o pack.tar.gz $URL
!mkdir -p /content/sp && tar -xzf /content/pack.tar.gz -C /content/sp --wildcards '*baseline/baseline.onnx' '*datasets/train.parquet' '*datasets/valid.parquet' 2>&1 | grep -v LIBARCHIVE
!rm -f /content/pack.tar.gz
!df -h /content | tail -1

ONNX = glob.glob('/content/sp/**/baseline/baseline.onnx', recursive=True)[0]
TRAIN_PQ = glob.glob('/content/sp/**/datasets/train.parquet', recursive=True)[0]
VALID_PQ = glob.glob('/content/sp/**/datasets/valid.parquet', recursive=True)[0]
KNOWN = '321ec67b93f3c5dde2c0901ca4532072dad5f65753c2e6519e93ea3555f6a6fd'   # baseline.onnx from your starter pack
same = hashlib.sha256(open(ONNX, 'rb').read()).hexdigest() == KNOWN
print('baseline.onnx:', 'the same file as in your starter pack' if same else 'DIFFERS from your starter-pack copy (organizers updated it?)')
shutil.copy(ONNX, W + '/baseline.onnx')
import pyarrow.parquet as pq
print('train sequences:', pq.ParquetFile(TRAIN_PQ).num_row_groups, '| valid sequences:', pq.ParquetFile(VALID_PQ).num_row_groups)

## 3. Валидация (~1.5 мин)
Те же 600 последовательностей, что и раньше (`valid_sub.parquet` с Drive), чтобы скоры сравнивались с прошлыми запусками.

In [ ]:
VSUB = W + '/colab_data/valid_sub.parquet'
if os.path.exists(VSUB):
    !python prepare_data.py --src $VSUB --out /content/data/valid
else:
    !python prepare_data.py --src $VALID_PQ --out /content/data/valid --max-seqs 600

## 4. Дообучение бейзлайна на всех данных (~1 ч)
Первая строка `VALID (start = initial weights)` — скор исходного бейзлайна на нашей валидации. **Он завышен:** на валидации бейзлайн давал 0.686, а на лидерборде 0.6505, тогда как у наших моделей разрыв ~0.015. Похоже, бейзлайн видел эти данные при обучении. Поэтому если валидация во время дообучения просядет, это ещё не значит, что на тесте модель стала хуже.

`data-wait` в логе — доля времени, когда GPU ждёт чтения данных; нормально, если меньше ~30%.

Если сессия оборвалась: ячейки 1–3, затем эта же ячейка с `--resume` в конце команды.

In [ ]:
RUN = W + '/runs/ft_baseline_full'
!python train.py --train-parquet $TRAIN_PQ --valid-dir /content/data/valid --out $RUN \
    --arch baseline --init-onnx $ONNX --keep-epochs \
    --epochs 6 --batch 32 --chunk 1000 --lr 3e-4 --mse-weight 0.1

## 5. Сабмит из последней эпохи + проверка
Почему последняя эпоха, а не `best.pt`: `best.pt` выбирается по валидации, а на ней исходный бейзлайн завышен — поэтому `best.pt`, скорее всего, так и останется исходными весами. Последняя эпоха — честный кандидат; сравниваем её на лидерборде с `baseline_x03` (0.6505). Все эпохи сохранены (`epoch_0.pt` … `epoch_5.pt`), так что можно отправить и промежуточную.

In [ ]:
import json
for e in json.load(open(RUN + '/history.json')):
    print(f"epoch {e['epoch']:2d}  WP(mask) {e['WP_mask_best']:.4f}  WP(all rows) {e['WP_all_at_best']:.4f}")
!python export.py --ckpt $RUN/last.pt --valid-dir /content/data/valid --out $W/submissions/ft_baseline_full
!python check_submission.py $W/submissions/ft_baseline_full.zip --data $VALID_PQ --seqs 5 --rows 20000
from google.colab import files
files.download(W + '/submissions/ft_baseline_full.zip')

## 6. (Пока сабмит считается) наша GRU на всех данных (~1 ч 20 мин)
Обучение с нуля, валидация здесь честная, поэтому берём `best.pt`. На 3500 последовательностях эта модель дала 0.644 на лидерборде, теперь данных втрое больше.

In [ ]:
RUN2 = W + '/runs/gru_full'
!python train.py --train-parquet $TRAIN_PQ --valid-dir /content/data/valid --out $RUN2 \
    --arch gru --epochs 8 --batch 64 --chunk 1000 --lr 2e-3 --keep-epochs
!python export.py --ckpt $RUN2/best.pt --valid-dir /content/data/valid --out $W/submissions/gru_full
!python check_submission.py $W/submissions/gru_full.zip --data $VALID_PQ --seqs 5 --rows 20000